## Zadanie 4: 

Wcześniej wytrenowaliśmy bazowy model `ConvNet` (przypominający architekturę LeNet-5 z 1998 roku). Na zbiorze CIFAR-10 osiąga on skuteczność rzędu 60%. To dobry start, ale nowoczesne sieci potrafią znacznie więcej.

Twoim zadaniem jest stworzenie nowej klasy `VGGLite`, która osiągnie **minimum 75% skuteczności** na zbiorze testowym. 

**Twój warsztat (co możesz zmienić/dodać):**
1. **Zasada VGG (Mniejsze filtry, głębsza sieć):** Zastąp duże filtry 5x5 mniejszymi filtrami 3x3. Zamiast jednej warstwy splotowej przed MaxPool, daj dwie (np. `Conv2d(3x3) -> ReLU -> Conv2d(3x3) -> ReLU -> MaxPool`). Aby nie tracić wymiarów przedwcześnie, użyj `padding=1`.
2. **Więcej kanałów (Grubsza sieć):** 6 i 16 kanałów to za mało na skomplikowane cechy (psy, samochody, statki). Spróbuj schematu: np. 32 kanały w pierwszym bloku, 64 w drugim.
3. **Regularyzacja (Lekarstwo na przeuczenie):** Jeśli sieć zacznie uczyć się na pamięć (loss treningowy spada, walidacyjny rośnie), dodaj `nn.BatchNorm2d` po warstwach splotowych oraz `nn.Dropout` (np. z prawdopodobieństwem 0.3 lub 0.5) przed warstwami `Linear`.

Poniżej znajduje się szkielet, od którego możesz zacząć. Uzupełnij go, przelicz wymiary spłaszczania i rozpocznij trening!

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class VGGLite(nn.Module):
    def __init__(self):
        super(VGGLite, self).__init__()
        
        # --- BLOK 1 ---
        # Zamiast jednego splotu 5x5, używamy dwóch splotów 3x3.
        # Wskazówka: Zwiększamy out_channels do np. 32. padding=1 utrzymuje wymiar 32x32.
        self.conv1_1 = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1)
        # TUTAJ MOŻESZ DODAĆ: nn.BatchNorm2d(32)
        self.conv1_2 = nn.Conv2d(in_channels=32, out_channels=32, kernel_size=3, padding=1)
        # TUTAJ MOŻESZ DODAĆ: nn.BatchNorm2d(32)
        
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2) # Zmniejsza wymiar z 32x32 na 16x16
        
        # --- BLOK 2 ---
        # Kolejny zestaw splotów. Zwiększ liczbę kanałów do np. 64.
        self.conv2_1 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.conv2_2 = nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, padding=1)
        
        # Kolejny pool (Zmniejszy wymiar z 16x16 na 8x8)
        
        # --- KLASYFIKATOR ---
        # TODO: Przelicz dokładnie ten wymiar!
        # Jeśli na końcu masz 64 kanały i obraz 8x8, to in_features = 64 * 8 * 8 = 4096
        self.fc1 = nn.Linear(in_features=..., out_features=512)
        # TUTAJ MOŻESZ DODAĆ: nn.Dropout(0.5)
        self.fc2 = nn.Linear(512, 10) # 10 klas na wyjściu CIFAR-10

    def forward(self, x):
        # Przepływ bloku 1
        x = F.relu(self.conv1_1(x))
        x = F.relu(self.conv1_2(x))
        x = self.pool(x)
        
        # Przepływ bloku 2
        x = F.relu(self.conv2_1(x))
        x = F.relu(self.conv2_2(x))
        x = self.pool(x)
        
        # Spłaszczanie
        # TODO: Wpisz prawidłową wartość
        x = x.view(-1, ...) 
        
        # Przepływ klasyfikatora
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        
        return x

# Szybki test wymiarów przed właściwym treningiem
dummy_cifar = torch.randn(1, 3, 32, 32)
test_model = VGGLite()
print("Kształt po przejściu przez VGGLite:", test_model(dummy_cifar).shape)

TypeError: empty(): argument 'size' failed to unpack the object at pos 2 with error "type must be tuple of ints,but got ellipsis"